In [23]:
import os
import numpy as np
import pandas as pd
import random
import torch
from torch import nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoderLayer, TransformerDecoder
import matplotlib.pyplot as plt
from scipy.signal import cont2discrete, lti, dlti, dstep
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import math
import scipy.integrate
import random

In [24]:
class TransformerAutoencoder(nn.Module):
    def __init__(self, 
                 encoder_input_dim, 
                 decoder_input_dim, 
                 hidden_dim,
                 num_heads, 
                 encoder_embedding_dim, 
                 decoder_embedding_dim,
                 num_layers, 
                 dropout):
        super(TransformerAutoencoder, self).__init__()
        self.encoder_input_dim = encoder_input_dim
        self.decoder_input_dim = decoder_input_dim
        self.encoder_embedding_dim = encoder_embedding_dim
        self.decoder_embedding_dim = decoder_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout

        # Encoder Embedding
        self.encoder_embedding = nn.Linear(self.encoder_input_dim, 
                                           self.encoder_embedding_dim)

        # Encoder
        self.encoder_layer = TransformerEncoderLayer(d_model=self.encoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.encoder = TransformerEncoder(self.encoder_layer,
                                          num_layers=self.num_layers)

        # Decoder Embedding
        self.decoder_embedding = nn.Linear(self.decoder_input_dim, 
                                           self.decoder_embedding_dim)

        # Decoder
        self.decoder_layer = TransformerDecoderLayer(d_model=self.decoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.decoder = TransformerDecoder(self.decoder_layer, 
                                          num_layers=self.num_layers)

        # Final output layer
        self.out = nn.Linear(self.decoder_embedding_dim,
                             self.decoder_input_dim)

    def forward(self, inputs, targets):        
        # Encode the input
        encoded_input = self.encoder(self.encoder_embedding(inputs))
        
        # Decode the target
        decoder_input = self.decoder_embedding(targets)
        decoded_target = self.decoder(decoder_input, encoded_input)        
        
        # Apply the final output layer
        target_output = self.out(decoded_target)
        return target_output

In [25]:
def data_generation(num_sequences, sequence_length, number_masses):
    dim_y = number_masses
    dim_x= 2*dim_y 

    X_data_array = np.empty((num_sequences, sequence_length, dim_x))
    Y_data_array = np.empty((num_sequences, sequence_length, dim_y))

    m = np.ones(dim_y)
    m = 10*m
   
    k = np.ones(dim_y)
    k = 800*k

    d = np.ones(dim_y)
    d = 6*d

    A_c = np.zeros((dim_x,dim_x))

    offset = 0
    for i in range(dim_x):
        if i % 2 == 0:
            A_c[i,i+1] = 1

        if i % 2 == 1:
            if i != dim_x-1:
                A_c[i,i-1] = -(k[i-1-offset]+k[i-offset])/m[i-1-offset]
                A_c[i,i] = -(d[i-1-offset]+d[i-offset])/m[i-1-offset]
                A_c[i,i+1] = k[i-offset]/m[i-1-offset]
                A_c[i,i+2] = d[i-offset]/m[i-1-offset]
            else:
                A_c[i,i-1] = -k[i-dim_y]/m[i-dim_y]
                A_c[i,i] = -d[i-dim_y]/m[i-dim_y]

            if i != 1:
                A_c[i,i-3] = k[i-1-offset]/m[i-1-offset]
                A_c[i,i-2] = d[i-1-offset]/m[i-1-offset]

            offset += 1

    B_c = np.zeros((dim_x,dim_y))

    H_c = np.zeros((dim_y,dim_x))
    offset = 0
    for i in range(dim_y):
        H_c[i,i+offset] = 1

        offset += 1

    D_c = np.array([[0.]])

   
    dt = 0.1 
    d_system = cont2discrete((A_c, B_c, H_c, D_c),dt)
    A = d_system[0] 
    H = d_system[2] 

    def is_schur(matrix):
       
        eigenvalues, _ = np.linalg.eig(matrix)
        if np.all(np.abs(eigenvalues) < 1):
            print(np.abs(eigenvalues))
            return True
        else:
            return False

    # if is_schur(A):
    #     print("The matrix is Schur.")
    # else:
    #     print("The matrix is not Schur.")


    sigma_p = 0.01 
    sigma_p_diag = (sigma_p**2)*np.ones(dim_x)
    Q = np.diag(sigma_p_diag)

    sigma_m = 0.01 
    sigma_m_diag = (sigma_m**2)*np.ones(dim_y)
    R = np.diag(sigma_m_diag)

    sigma_x = 0.01 
    sigma_x_diag = (sigma_p**2)*np.ones(dim_x)
    P = np.diag(sigma_x_diag)


    for s in range(num_sequences):
        mu_x0 = np.random.uniform(-10,10,size=dim_x) 
        x=np.random.multivariate_normal(mu_x0,P)
       
        X_data_array[s,0,:] = np.squeeze(np.asarray(x))

       
        v_0 = np.random.multivariate_normal(np.zeros(dim_y),R).reshape(-1,1)

        y = H.dot(x.reshape(-1,1)) + v_0


        W = np.random.multivariate_normal(np.zeros(dim_x), Q, sequence_length)
        V = np.random.multivariate_normal(np.zeros(dim_y), R, sequence_length)
        Y_data_array[s,0,:] = np.squeeze(np.asarray(y))

        for t in range(1,sequence_length+1):
            w = W[t-1].reshape(-1,1) 
            v = V[t-1].reshape(-1,1)
        
            x = A.dot(x.reshape(-1,1)) + w 
            y = H.dot(x.reshape(-1,1)) + v 

            X_data_array[s,t:t+1,:] = x.T
            Y_data_array[s,t:t+1,:] = y.T

    return X_data_array, Y_data_array, A, H

print(data_generation(200, 100, 2)[0].shape)

(200, 100, 4)


In [26]:
def data_loaders(X_np, Y_np, batch_size, train_ratio, val_ratio):
   
  
    X = torch.tensor(X_np, dtype=torch.float32).unsqueeze(-1)  
    Y = torch.tensor(Y_np, dtype=torch.float32).unsqueeze(-1)  

    dataset = TensorDataset(Y, X)

    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = int(val_ratio * total_size)
    test_size = total_size - train_size - val_size

    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

    train_data = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_data = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_data = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_data, val_data, test_data

    

In [ ]:
def train_kalmanformer(model, data_loader, optimizer, epochs):
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    model.train()
    

    for epoch in range(epochs):
        total_loss = 0
        for y_seq, x_true in data_loader:
            # B, T, N, D = x_true.shape
            # x_true = x_true.view(B, T, N * D)

            # _, T, n, _ = y_seq.shape

            # x0 = torch.zeros_like(x_true[:, 0, :, :])
            # model.InitSequence(x0, T)

#             g, i, u = model(y_seq[:, 1:], return_gain=True)
#      
#             K = torch.stack(g, dim=1)          
#             dz = torch.stack(i, dim=1)    
#             dx = torch.stack(u, dim=1)       
#             pred_dx = torch.matmul(K, dz)           
           
#             loss = F.mse_loss(pred_dx, dx)    
       
            preds = model(y_seq[:, 1:,:],x_true[:,1:,:])  

            loss = F.mse_loss(preds, x_true[:, 1:, :, :]) 

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        print(f"Epoch {epoch + 1}: Loss = {total_loss:.6f}")


In [28]:

X_np, Y_np, A, H = data_generation(num_sequences=200, sequence_length = 500, number_masses = 1)

A = torch.tensor(A, dtype=torch.float32)
H = torch.tensor(H, dtype=torch.float32)

def f(x): return A @ x
def h(x): return H @ x

numSequence = X_np.shape[0]
encoder_input_dim = Y_np.shape[2]
decoder_input_dim = X_np.shape[2]
    
alpha = 0.81
num_layers = 8
num_epochs = 100
hidden_dim = 10
num_heads = 4
encoder_embedding_dim = 64
decoder_embedding_dim = 64
learn_rate = 0.1
weight_decay = 0.005
dropout = 0.05


model = TransformerAutoencoder(encoder_input_dim, 
                               decoder_input_dim,
                               hidden_dim, 
                               num_heads, 
                               encoder_embedding_dim,
                               decoder_embedding_dim, 
                               num_layers, 
                               dropout)

train_data, val_data, test_data = data_loaders(X_np, Y_np, batch_size=60, train_ratio =0.8, val_ratio =0.1)

optimizer = torch.optim.SGD(model.parameters(), lr=learn_rate, weight_decay = weight_decay)
train_kalmanformer(model, train_data, optimizer, epochs = 20)

AssertionError: query should be unbatched 2D or batched 3D tensor but received 4-D query tensor

In [ ]:
def evaluate_true_mse(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_seqs = 0

    for batch in data_loader:
        y_seq, x_true = batch  
#         b, T, m, _ = x_true.shape
#         x0 = torch.zeros_like(x_true[:, 0, :, :])  
# #         x0 = x_true[:, 0, :, :]  
#         model.InitSequence(x0, T)

        x_pred = model(y_seq[:, 1:], x_true[:,1:])   
    
        x_gt = x_true[:, 1:, :, :]           
#         print(x_pred[0] - x_gt[0])
        loss = F.mse_loss(x_pred, x_gt)  
        total_loss += loss.item()
#         total_seqs += b  


    final_mse = total_loss 
    print(f"[MSE] = {final_mse:.6f}")
    return final_mse

In [ ]:
evaluate_true_mse(model, test_data)


In [ ]:
def plot_pred_vs_true_simple(model, batch, title_prefix="Train"):
    model.eval()
    y_seq, x_true = batch
#     x0 = torch.zeros_like(x_true[:, 0, :, :])  
# #     x0 = x_true[:, 0, :, :]
#     T = y_seq.shape[1]

#     model.InitSequence(x0, T)
    
    with torch.no_grad():
        x_pred = model(y_seq[:, 1:], x_true[:,1:]) 

    x_true = x_true[:, 1:, :, :]     

    idx = 5
    pred_np = x_pred[idx].squeeze(-1).cpu().numpy()  
    true_np = x_true[idx].squeeze(-1).cpu().numpy()  

    time = np.arange(pred_np.shape[0])
    for state in range(pred_np.shape[1]-1):
        plt.plot(time, true_np[:, state], label=f'True State {idx}')
        plt.plot(time, pred_np[:, state], '--', label=f'Pred State {idx}')

    plt.title(f'{title_prefix} Trajectory')
    plt.xlabel('Time')
    plt.ylabel('State')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:

# plot_pred_vs_true_simple(model, next(iter(train_data)), title_prefix="Train")
plot_pred_vs_true_simple(model, next(iter(test_data)), title_prefix="Test")